# Fine-Tune NVIDIA Nemotron-3-Super-120B on train2earn Data

This notebook fine-tunes NVIDIA Nemotron-3-Super-120B using the Solana/Clawd training datasets uploaded to GCS.

**Base model:** `nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16`

**Training data source:** `gs://knowledge-base-docs-x402-477302/datasets/sft/` from the train2earn project.

**Requirements:**
- 4x H100 GPUs (≥264 GB VRAM for BF16 full fine-tune)
- For QLoRA: 1-2x H100 (~48 GB VRAM)
- NVIDIA Container Toolkit
- GCS access (service account with `storage.objectViewer` on the datasets bucket)

In [ ]:
# Install dependencies
!pip install -q torch==2.9.1 transformers==4.49.0 accelerate==1.5.2 datasets==3.5.0
!pip install -q peft==0.14.0 bitsandbytes==0.45.5 trl==0.16.0
!pip install -q google-cloud-storage wandb

In [ ]:
import json
import os
import torch
from google.cloud import storage
from datasets import Dataset, DatasetDict, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import warnings
warnings.filterwarnings('ignore')

## 1. Load Training Data from GCS

In [ ]:
# Configuration
BUCKET_NAME = "x402-477302-train2earn-datasets"
BUCKET_PUBLIC = "x402-477302-train2earn-datasets-public"
BASE_MODEL = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16"
OUTPUT_DIR = "/ephemeral/nemotron-clawd-finetune"

# Datasets to load (from GCS)
DATASET_FILES = [
    "sft/nemo_clawd_master_sft.jsonl",      # ~5.3 MiB — primary Clawd+Nemotron
    "sft/nemo_clawd_combined_sft.jsonl",    # ~650 KiB — combined SFT
    "sft/nvidia_trading_factory_sft.jsonl", # ~626 KiB — trading factory
    "sft/solana_clawd_merged.jsonl",        # ~41 MiB — Clawd Solana merged
    "sft/core_ai_clawd_sft.jsonl",          # ~96 MiB — core AI
    "sft/clawd_fable_sft.jsonl",           # ~13 MiB — Clawd fable
]

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
def load_jsonl_from_gcs(bucket_name: str, blob_path: str) -> list:
    """Load a JSONL file from GCS into a list of dicts."""
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_path)
    content = blob.download_as_string().decode('utf-8')
    return [json.loads(line) for line in content.strip().split('\n') if line]

def format_chat(example: dict) -> dict:
    """Format a training example into chat template format."""
    if "messages" in example:
        return {"text": example["messages"]}
    elif "instruction" in example and "output" in example:
        return {
            "text": [
                {"role": "user", "content": example["instruction"]},
                {"role": "assistant", "content": example["output"]}
            ]
        }
    elif "prompt" in example and "completion" in example:
        return {
            "text": [
                {"role": "user", "content": example["prompt"]},
                {"role": "assistant", "content": example["completion"]}
            ]
        }
    else:
        # Try to infer from first available keys
        keys = list(example.keys())
        if len(keys) >= 2:
            return {
                "text": [
                    {"role": "user", "content": str(example[keys[0]])},
                    {"role": "assistant", "content": str(example[keys[1]])}
                ]
            }
    return {"text": [{"role": "user", "content": str(example)}]}

all_records = []
for dataset_file in DATASET_FILES:
    print(f"Loading {dataset_file}...")
    try:
        records = load_jsonl_from_gcs(BUCKET_NAME, dataset_file)
        formatted = [format_chat(r) for r in records]
        all_records.extend(formatted)
        print(f"  -> {len(records)} records loaded")
    except Exception as e:
        print(f"  -> Error: {e}")

print(f"\nTotal records: {len(all_records)}")

In [ ]:
# Create HuggingFace dataset
dataset = Dataset.from_list(all_records)

# Train/eval split
split_dataset = dataset.train_test_split(test_size=0.05, seed=42)
dataset_dict = DatasetDict({
    "train": split_dataset["train"],
    "eval": split_dataset["test"],
})

print(f"Train: {len(dataset_dict['train'])} examples")
print(f"Eval: {len(dataset_dict['eval'])} examples")
print(f"\nSample:\n{dataset_dict['train'][0]['text'][:2]}")

## 2. Load Tokenizer & Model

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    use_fast=True,
)

# Set padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Vocab size: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token}")
print(f"EOS token: {tokenizer.eos_token}")

In [ ]:
# Quantization config for QLoRA (use 4-bit to fit on fewer GPUs)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading model (this will take several minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto",
    use_cache=False,
)

print(f"Model loaded: {model.config.model_type}")
print(f"Params: {model.num_parameters():,}")

## 3. Setup LoRA Configuration

In [ ]:
# LoRA config — fine-tune attention + MLP layers
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

# Enable gradient checkpointing
model.config.use_cache = False

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(model.print_trainable_parameters())

## 4. Train

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    bf16=True,
    tf32=True,
    max_grad_norm=0.3,
    report_to="wandb",
    run_name="nemotron-clawd-finetune",
    ddp_find_unused_parameters=False,
)

In [ ]:
def formatting_func(example):
    """Format chat messages into tokenizer's chat template."""
    text = tokenizer.apply_chat_template(
        example["text"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

# Apply formatting
formatted_train = dataset_dict["train"].map(formatting_func)
formatted_eval = dataset_dict["eval"].map(formatting_func)

print("Formatted sample:")
print(formatted_train[0]["text"][:500])

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=formatted_train,
    eval_dataset=formatted_eval,
    max_seq_length=4096,
    dataset_text_field="text",
    packing=True,
)

In [ ]:
# Train!
trainer.train()

## 5. Save & Upload Adapters

In [ ]:
# Save LoRA adapters locally
adapter_path = f"{OUTPUT_DIR}/final_adapter"
trainer.save_model(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved to {adapter_path}")

# Upload to GCS
from google.cloud import storage

client = storage.Client()
bucket = client.bucket(BUCKET_NAME)

for root, dirs, files in os.walk(adapter_path):
    for file in files:
        local_path = os.path.join(root, file)
        rel_path = os.path.relpath(local_path, adapter_path)
        blob = bucket.blob(f"adapters/nemotron-clawd-finetune/{rel_path}")
        blob.upload_from_filename(local_path)
        print(f"  Uploaded: adapters/nemotron-clawd-finetune/{rel_path}")

## 6. Inference Test

In [ ]:
from peft import PeftModel

# Reload base model and merge adapter
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto",
)

merged_model = PeftModel.from_pretrained(base_model, adapter_path)
merged_model = merged_model.merge_and_unload()

# Test inference
messages = [
    {"role": "system", "content": "You are a Clawd agent specialized in Solana blockchain analytics and DeFi trading."},
    {"role": "user", "content": "Analyze the current Solana mempool for arbitrage opportunities between Jupiter and Phoenix markets."},
]

tokenized = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to(merged_model.device)

outputs = merged_model.generate(
    tokenized,
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.95,
    do_sample=True,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))